In [0]:
from pyspark.sql.functions import explode, col
import pandas as pd

# Read the Bronze Delta table
bronze_df = spark.table("workspace.default.bronze_teams")

# Check the number of rows in Bronze
print("Bronze records:", bronze_df.count())

# Inspect the schema
bronze_df.printSchema()

# Convert the teams array into individual rows
teams_df = bronze_df.select(
    explode(col("teams")).alias("team")
)

# Display the result
display(teams_df)

# Count the individual team records
print("Total teams:", teams_df.count())

# Select and rename the team attributes we need
silver_df = teams_df.select(
    col("team.id").alias("team_id"),
    col("team.name").alias("team_name"),
    col("team.shortName").alias("short_name"),
    col("team.tla").alias("team_code"),
    col("team.venue").alias("stadium"),
    col("team.founded").alias("founded_year")
)

# Preview the structured dataset
display(silver_df)

# Removing any duplicates from team_id
silver_df = silver_df.dropDuplicates(["team_id"])
silver_df = silver_df.dropna(subset=["team_id"])
print("Clean team records:", silver_df.count())
display(silver_df)

# Validate the cleaned team dataset
# Check that the dataset contains records
assert silver_df.count() > 0, "Dataset is empty"

# Check that every team has a unique ID
assert (
    silver_df.select("team_id").distinct().count()
    == silver_df.count()
), "Duplicate team IDs found"

# Check that team IDs are not null
assert (
    silver_df.filter("team_id IS NULL").count() == 0
), "Missing team IDs found"

print("All data quality checks passed!")

# Destination for the cleaned Silver dataset
table_name = "workspace.default.silver_teams"

# Save the validated DataFrame as a Delta table
(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(table_name)
)

print(f"Silver table created: {table_name}")

# Read the saved Silver Delta table
saved_df = spark.table("workspace.default.silver_teams")

# Confirm the data was saved successfully
print("Saved records:", saved_df.count())

display(saved_df)

